# Kaggle Pipeline - Assemble `ready-data`

Notebook nay dung de gom du lieu tu cac Kaggle datasets/input hien co thanh 1 package thong nhat `ready-data/`.

## Muc tieu
- Gom du lieu cho train va demo vao cung 1 cau truc thu muc
- Copy nhanh metadata, clips, interactions, sequences, metrics, checkpoint
- Tao san cac file `demo/` de web demo dung duoc ngay
- Tao `README.md` va `manifest.json` de theo doi package

## Input mong doi
- `ready-items` dataset: `publish/ready_items_v1` + `outputs/clips`
- `mock-user-interactions` dataset: profiles + interactions + sasrec sequences
- `sasrec-base-train` dataset hoac `sasrec_outputs` trong `/kaggle/working/`
- `podtok-data` la input optional cho phase sau

## Output
- `/kaggle/working/ready-data/`
- co the publish thanh 1 dataset moi sau khi notebook chay xong


In [ ]:
%pip install -q orjson


In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import shutil
from typing import Dict, Any, List

import orjson
import pandas as pd

WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "ready-data"

READY_ITEMS_DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/sonisson/ready-items"),
    WORK_DIR,
]
READY_ITEMS_PUBLISH_CANDIDATES = [
    Path("/kaggle/input/datasets/sonisson/ready-items/publish/ready_items_v1"),
    WORK_DIR / "publish" / "ready_items_v1",
]
MOCK_OUTPUTS_CANDIDATES = [
    Path("/kaggle/input/datasets/sonisson/mock-user-interactions/mock_outputs"),
    WORK_DIR / "mock_outputs",
]
SASREC_OUTPUTS_CANDIDATES = [
    Path("/kaggle/input/datasets/sonisson/sasrec-base-train/sasrec_outputs"),
    WORK_DIR / "sasrec_outputs",
]
PODTOK_DATA_CANDIDATES = [
    Path("/kaggle/input/datasets/sonisson/podtok-data"),
]

def resolve_existing(candidates: List[Path]) -> Path | None:
    for path in candidates:
        if path.exists():
            return path
    return None

READY_ITEMS_DATASET_ROOT = resolve_existing(READY_ITEMS_DATASET_CANDIDATES)
READY_ITEMS_ROOT = resolve_existing(READY_ITEMS_PUBLISH_CANDIDATES)
MOCK_OUTPUTS_ROOT = resolve_existing(MOCK_OUTPUTS_CANDIDATES)
SASREC_OUTPUTS_ROOT = resolve_existing(SASREC_OUTPUTS_CANDIDATES)
PODTOK_DATA_ROOT = resolve_existing(PODTOK_DATA_CANDIDATES)

CLIP_SOURCE_CANDIDATES = [
    READY_ITEMS_DATASET_ROOT / "outputs" / "clips" if READY_ITEMS_DATASET_ROOT else None,
    READY_ITEMS_DATASET_ROOT / "clips" if READY_ITEMS_DATASET_ROOT else None,
    WORK_DIR / "outputs" / "clips",
    WORK_DIR / "clips",
]
CLIP_SOURCE_ROOT = resolve_existing([path for path in CLIP_SOURCE_CANDIDATES if path is not None])

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for rel_dir in [
    "catalog",
    "assets/clips",
    "interactions/mock",
    "interactions/real",
    "training/sasrec",
    "training/features",
    "demo/users",
    "demo/feed",
    "demo/recommendations",
    "demo/ui",
    "reports",
    "schemas",
]:
    (OUTPUT_ROOT / rel_dir).mkdir(parents=True, exist_ok=True)

print("READY_ITEMS_DATASET_ROOT:", READY_ITEMS_DATASET_ROOT)
print("READY_ITEMS_ROOT:", READY_ITEMS_ROOT)
print("MOCK_OUTPUTS_ROOT:", MOCK_OUTPUTS_ROOT)
print("SASREC_OUTPUTS_ROOT:", SASREC_OUTPUTS_ROOT)
print("PODTOK_DATA_ROOT:", PODTOK_DATA_ROOT)
print("CLIP_SOURCE_ROOT:", CLIP_SOURCE_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


In [ ]:
copied_files = []
missing_files = []
source_map = {}

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(orjson.loads(line))
    return rows

def write_json(data: Dict[str, Any], path: Path):
    with open(path, "wb") as f:
        f.write(orjson.dumps(data, option=orjson.OPT_INDENT_2))

def write_jsonl(records: List[Dict[str, Any]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        for row in records:
            f.write(orjson.dumps(row))
            f.write(b"\n")

def write_text(text: str, path: Path):
    path.write_text(text, encoding="utf-8")

def copy_file(src: Path | None, dst: Path, label: str):
    if src is None or not src.exists():
        missing_files.append({"label": label, "expected_path": str(src) if src else "None", "dst": str(dst)})
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied_files.append(str(dst.relative_to(OUTPUT_ROOT)))
    source_map[str(dst.relative_to(OUTPUT_ROOT))] = str(src)
    return True

def copy_tree(src: Path | None, dst: Path, label: str):
    if src is None or not src.exists():
        missing_files.append({"label": label, "expected_path": str(src) if src else "None", "dst": str(dst)})
        return 0
    count = 0
    for path in src.rglob("*"):
        if path.is_file():
            rel = path.relative_to(src)
            target = dst / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
            copied_files.append(str(target.relative_to(OUTPUT_ROOT)))
            source_map[str(target.relative_to(OUTPUT_ROOT))] = str(path)
            count += 1
    return count

ready_item_mapping = [
    (READY_ITEMS_ROOT / "item_metadata_final.jsonl" if READY_ITEMS_ROOT else None, OUTPUT_ROOT / "catalog" / "item_metadata_final.jsonl", "item_metadata_final_jsonl"),
    (READY_ITEMS_ROOT / "item_metadata_final.csv" if READY_ITEMS_ROOT else None, OUTPUT_ROOT / "catalog" / "item_metadata_final.csv", "item_metadata_final_csv"),
    (READY_ITEMS_ROOT / "dataset_manifest.json" if READY_ITEMS_ROOT else None, OUTPUT_ROOT / "reports" / "ready_items_dataset_manifest.json", "ready_items_dataset_manifest"),
    (READY_ITEMS_ROOT / "final_dataset_summary.json" if READY_ITEMS_ROOT else None, OUTPUT_ROOT / "reports" / "clip_generation_summary.json", "clip_generation_summary"),
    (READY_ITEMS_ROOT / "final_dedup_summary.json" if READY_ITEMS_ROOT else None, OUTPUT_ROOT / "reports" / "dedup_summary.json", "dedup_summary"),
    (READY_ITEMS_ROOT / "final_clip_qa_summary.json" if READY_ITEMS_ROOT else None, OUTPUT_ROOT / "reports" / "qa_summary.json", "qa_summary"),
]
for src, dst, label in ready_item_mapping:
    copy_file(src, dst, label)

mock_mapping = [
    (MOCK_OUTPUTS_ROOT / "item_id_map.json" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "catalog" / "item_id_map.json", "item_id_map"),
    (MOCK_OUTPUTS_ROOT / "mock_user_profiles.jsonl" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "interactions" / "mock" / "user_profiles.jsonl", "mock_user_profiles"),
    (MOCK_OUTPUTS_ROOT / "user_interactions.jsonl" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "interactions" / "mock" / "user_interactions.jsonl", "user_interactions_jsonl"),
    (MOCK_OUTPUTS_ROOT / "user_interactions.parquet" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "interactions" / "mock" / "user_interactions.parquet", "user_interactions_parquet"),
    (MOCK_OUTPUTS_ROOT / "interaction_summary.json" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "interactions" / "mock" / "interaction_summary.json", "interaction_summary"),
    (MOCK_OUTPUTS_ROOT / "sasrec_train_sequences.jsonl" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "train_sequences.jsonl", "sasrec_train_sequences"),
    (MOCK_OUTPUTS_ROOT / "sasrec_valid_sequences.jsonl" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "valid_sequences.jsonl", "sasrec_valid_sequences"),
    (MOCK_OUTPUTS_ROOT / "sasrec_test_sequences.jsonl" if MOCK_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "test_sequences.jsonl", "sasrec_test_sequences"),
]
for src, dst, label in mock_mapping:
    copy_file(src, dst, label)

sasrec_mapping = [
    (SASREC_OUTPUTS_ROOT / "train_history.json" if SASREC_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "train_history.json", "train_history"),
    (SASREC_OUTPUTS_ROOT / "valid_metrics.json" if SASREC_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "valid_metrics.json", "valid_metrics"),
    (SASREC_OUTPUTS_ROOT / "test_metrics.json" if SASREC_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "test_metrics.json", "test_metrics"),
    (SASREC_OUTPUTS_ROOT / "model_state.pt" if SASREC_OUTPUTS_ROOT else None, OUTPUT_ROOT / "training" / "sasrec" / "model_state.pt", "model_state"),
]
for src, dst, label in sasrec_mapping:
    copy_file(src, dst, label)

clip_file_count = copy_tree(CLIP_SOURCE_ROOT, OUTPUT_ROOT / "assets" / "clips", "clips_folder")

write_text("Real interactions will be added later.\n", OUTPUT_ROOT / "interactions" / "real" / "README.md")
write_text("Feature tables and embeddings can be added later.\n", OUTPUT_ROOT / "training" / "features" / "README.md")
write_text("JSON schema files can be added later when contracts are frozen.\n", OUTPUT_ROOT / "schemas" / "README.md")

print("copied_files:", len(copied_files))
print("missing_files:", len(missing_files))
print("clip_file_count:", clip_file_count)


In [ ]:
item_metadata_path = OUTPUT_ROOT / "catalog" / "item_metadata_final.jsonl"
item_metadata_csv_path = OUTPUT_ROOT / "catalog" / "item_metadata_final.csv"
item_id_map_path = OUTPUT_ROOT / "catalog" / "item_id_map.json"
profiles_path = OUTPUT_ROOT / "interactions" / "mock" / "user_profiles.jsonl"
interactions_path = OUTPUT_ROOT / "interactions" / "mock" / "user_interactions.jsonl"

item_rows = read_jsonl(item_metadata_path) if item_metadata_path.exists() else []
profile_rows = read_jsonl(profiles_path) if profiles_path.exists() else []
interaction_rows = read_jsonl(interactions_path) if interactions_path.exists() else []
item_id_map = orjson.loads(item_id_map_path.read_bytes()) if item_id_map_path.exists() else {}

item_df = pd.DataFrame(item_rows) if item_rows else pd.DataFrame()
profiles_df = pd.DataFrame(profile_rows) if profile_rows else pd.DataFrame()
interactions_df = pd.DataFrame(interaction_rows) if interaction_rows else pd.DataFrame()

def normalize_clip_audio_path(path_value: Any) -> Any:
    if path_value is None:
        return None
    path_text = str(path_value).replace("\\", "/").strip()
    if not path_text:
        return path_text
    for prefix in ["outputs/clips/", "clips/", "assets/clips/"]:
        if path_text.startswith(prefix):
            path_text = path_text[len(prefix):]
            break
    return f"assets/clips/{path_text}"

if not item_df.empty:
    if "clip_audio_path" in item_df.columns:
        item_df["clip_audio_path"] = item_df["clip_audio_path"].apply(normalize_clip_audio_path)

    if item_id_map and "clip_id" in item_df.columns:
        item_df["item_id"] = item_df["clip_id"].map(lambda clip_id: item_id_map.get(str(clip_id)))

    preferred_cols = [
        "item_id", "clip_id", "episode_id", "title", "host", "keyword",
        "source_url", "source_audio_file", "clip_audio_path", "start_sec", "end_sec",
        "duration_sec", "start_segment_idx", "end_segment_idx", "num_segments",
        "transcript_text", "is_sentence_complete", "heuristic_score", "heuristic_reasons",
        "llm_score", "llm_label", "llm_reason", "qa_label", "qa_score", "qa_reason",
        "final_status", "dedup_status", "dedup_duplicate_of", "dedup_reason", "dedup_similarity"
    ]
    ordered_cols = [col for col in preferred_cols if col in item_df.columns] + [col for col in item_df.columns if col not in preferred_cols]
    item_df = item_df[ordered_cols].copy()

    if "item_id" in item_df.columns:
        item_df["item_id"] = item_df["item_id"].astype("Int64")

    item_rows = item_df.where(pd.notnull(item_df), None).to_dict(orient="records")
    write_jsonl(item_rows, item_metadata_path)
    item_df.to_csv(item_metadata_csv_path, index=False)

    feed_cols = [
        col for col in [
            "item_id", "clip_id", "episode_id", "title", "host", "keyword",
            "duration_sec", "clip_audio_path", "start_sec", "end_sec",
            "llm_score", "qa_score", "transcript_text"
        ] if col in item_df.columns
    ]
    feed_df = item_df[feed_cols].copy()
    sort_cols = [col for col in ["qa_score", "llm_score"] if col in feed_df.columns]
    if sort_cols:
        feed_df = feed_df.sort_values(sort_cols, ascending=[False] * len(sort_cols), na_position="last")
    feed_items = feed_df.where(pd.notnull(feed_df), None).to_dict(orient="records")
    featured_items = feed_items[:20]
    catalog_clip_ids = set(str(clip_id) for clip_id in item_df["clip_id"].dropna().tolist()) if "clip_id" in item_df.columns else set()
else:
    feed_items = []
    featured_items = []
    catalog_clip_ids = set()

demo_users = []
if not profiles_df.empty:
    sample_profiles = profiles_df.head(8).to_dict(orient="records")
    for idx, row in enumerate(sample_profiles, start=1):
        demo_users.append({
            "user_id": row.get("user_id", f"demo_user_{idx:03d}"),
            "display_name": f"Demo User {idx:02d}",
            "fav_keywords": row.get("fav_keywords", []),
            "fav_hosts": row.get("fav_hosts", []),
            "patience_level": row.get("patience_level"),
            "explore_rate": row.get("explore_rate"),
        })

demo_user_ids = {row["user_id"] for row in demo_users}
demo_histories = []
if not interactions_df.empty and demo_user_ids:
    interactions_df = interactions_df.sort_values(["user_id", "event_time"])
    for user_id in demo_user_ids:
        user_df = interactions_df[interactions_df["user_id"] == user_id].copy()
        if "watch_label" in user_df.columns:
            user_df = user_df[user_df["watch_label"] == 1]
        user_df = user_df.tail(10)
        history_cols = [
            col for col in [
                "event_time", "item_id", "clip_id", "episode_id", "watch_time_sec",
                "completion_rate", "is_like", "is_save", "is_share"
            ] if col in user_df.columns
        ]
        demo_histories.append({
            "user_id": user_id,
            "history": user_df[history_cols].where(pd.notnull(user_df[history_cols]), None).to_dict(orient="records")
        })

popularity_map = defaultdict(float)
if not interactions_df.empty:
    score_cols = [col for col in ["engagement_score", "watch_label", "is_like", "is_save", "is_share"] if col in interactions_df.columns]
    if score_cols and "item_id" in interactions_df.columns:
        tmp_df = interactions_df.copy()
        tmp_df["pop_score"] = 0.0
        if "engagement_score" in tmp_df.columns:
            tmp_df["pop_score"] += tmp_df["engagement_score"].fillna(0.0)
        if "watch_label" in tmp_df.columns:
            tmp_df["pop_score"] += tmp_df["watch_label"].fillna(0.0) * 0.5
        if "is_like" in tmp_df.columns:
            tmp_df["pop_score"] += tmp_df["is_like"].fillna(0.0) * 0.3
        if "is_save" in tmp_df.columns:
            tmp_df["pop_score"] += tmp_df["is_save"].fillna(0.0) * 0.3
        if "is_share" in tmp_df.columns:
            tmp_df["pop_score"] += tmp_df["is_share"].fillna(0.0) * 0.4
        pop_df = tmp_df.groupby("item_id", as_index=False)["pop_score"].mean()
        popularity_map = {int(row["item_id"]): float(row["pop_score"]) for _, row in pop_df.iterrows()}

demo_recommendations = []
if not item_df.empty and demo_users:
    item_records = item_df.where(pd.notnull(item_df), None).to_dict(orient="records")
    history_seen = {row["user_id"]: {evt.get("item_id") for evt in row.get("history", []) if evt.get("item_id") is not None} for row in demo_histories}
    for user in demo_users:
        seen_items = history_seen.get(user["user_id"], set())
        scored_items = []
        for item in item_records:
            item_id = item.get("item_id")
            if item_id in seen_items:
                continue
            keyword_match = 1.0 if item.get("keyword") in set(user.get("fav_keywords", [])) else 0.0
            host_match = 1.0 if item.get("host") in set(user.get("fav_hosts", [])) else 0.0
            qa_score = float(item.get("qa_score", 0.0) or 0.0)
            llm_score = float(item.get("llm_score", 0.0) or 0.0)
            pop_score = float(popularity_map.get(int(item_id), 0.0)) if item_id is not None else 0.0
            final_score = 1.2 * keyword_match + 0.8 * host_match + 0.15 * qa_score + 0.10 * llm_score + 0.50 * pop_score
            reason_parts = []
            if keyword_match:
                reason_parts.append("keyword_match")
            if host_match:
                reason_parts.append("host_match")
            if qa_score >= 8:
                reason_parts.append("high_qa")
            if pop_score >= 0.8:
                reason_parts.append("popular")
            scored_items.append({
                "item_id": item_id,
                "clip_id": item.get("clip_id"),
                "title": item.get("title"),
                "host": item.get("host"),
                "keyword": item.get("keyword"),
                "clip_audio_path": item.get("clip_audio_path"),
                "duration_sec": item.get("duration_sec"),
                "score": round(final_score, 4),
                "why_recommended": ", ".join(reason_parts) if reason_parts else "fallback_rank",
            })
        scored_items = sorted(scored_items, key=lambda x: x["score"], reverse=True)[:10]
        demo_recommendations.append({
            "user_id": user["user_id"],
            "items": scored_items,
        })

write_json({"items": feed_items}, OUTPUT_ROOT / "demo" / "feed" / "feed_items.json")
write_json({"items": featured_items}, OUTPUT_ROOT / "demo" / "feed" / "featured_items.json")
write_json({"users": demo_users}, OUTPUT_ROOT / "demo" / "users" / "demo_users.json")
write_json({"histories": demo_histories}, OUTPUT_ROOT / "demo" / "users" / "demo_user_history.json")
write_json({"recommendations": demo_recommendations}, OUTPUT_ROOT / "demo" / "recommendations" / "demo_recommendations.json")

demo_config = {
    "default_user_id": demo_users[0]["user_id"] if demo_users else None,
    "feed_file": "demo/feed/feed_items.json",
    "featured_file": "demo/feed/featured_items.json",
    "users_file": "demo/users/demo_users.json",
    "histories_file": "demo/users/demo_user_history.json",
    "recommendations_file": "demo/recommendations/demo_recommendations.json",
    "catalog_file": "catalog/item_metadata_final.jsonl",
    "clips_root": "assets/clips",
}
write_json(demo_config, OUTPUT_ROOT / "demo" / "ui" / "demo_config.json")

print("catalog_items:", len(item_rows))
print("catalog_items_with_item_id:", sum(1 for row in item_rows if row.get("item_id") is not None))
print("catalog_unique_clip_ids:", len(catalog_clip_ids))
print("demo_users:", len(demo_users))
print("demo_histories:", len(demo_histories))
print("demo_recommendations:", len(demo_recommendations))


In [ ]:
training_summary = {}
valid_metrics_path = OUTPUT_ROOT / "training" / "sasrec" / "valid_metrics.json"
test_metrics_path = OUTPUT_ROOT / "training" / "sasrec" / "test_metrics.json"
if valid_metrics_path.exists():
    training_summary["valid_metrics"] = orjson.loads(valid_metrics_path.read_bytes())
if test_metrics_path.exists():
    training_summary["test_metrics"] = orjson.loads(test_metrics_path.read_bytes())
if training_summary:
    write_json(training_summary, OUTPUT_ROOT / "reports" / "training_summary.json")

extra_clip_files = max(clip_file_count - len(item_rows), 0)

demo_readme_text = """# demo

Thu muc nay chua cac file JSON san cho web demo.

## Nen bat dau tu dau?
- Doc `ui/demo_config.json` truoc.
- Sau do load `users`, `histories`, `feed`, `recommendations` theo cac path trong file config.

## Quy tac dung clips
- `clip_audio_path` la path tuong doi tinh tu root `ready-data/`.
- Khong nen quet toan bo `assets/clips/` de render feed.
- Hay render item tu `catalog/` hoac `demo/feed/`, sau do join audio bang `clip_audio_path`.
"""
write_text(demo_readme_text, OUTPUT_ROOT / "demo" / "README.md")

readme_text = f"""# ready-data

Package nay gom du lieu da san sang cho ca train va demo.

## Thu muc chinh
- `catalog/`: source-of-truth cho item va clip metadata
- `assets/clips/`: audio clips de demo web
- `interactions/mock/`: mock users va interaction logs
- `training/sasrec/`: sequences, metrics, checkpoint baseline
- `demo/`: cac file JSON de web demo doc truc tiep
- `reports/`: summary de check nhanh package

## Neu ban lam web demo
- Bat dau tu `demo/ui/demo_config.json`
- Dung `demo/feed/feed_items.json` de render feed
- Dung `demo/users/demo_users.json` va `demo/users/demo_user_history.json` de mo phong user
- Dung `demo/recommendations/demo_recommendations.json` de render ket qua goi y
- `clip_audio_path` la duong dan tuong doi tinh tu root `ready-data/`

## Neu ban lam model/train
- Bat dau tu `catalog/item_metadata_final.jsonl`
- Dung `catalog/item_id_map.json` de map `clip_id -> item_id`
- Dung `interactions/mock/user_interactions.jsonl` cho event log
- Dung `training/sasrec/train_sequences.jsonl`, `valid_sequences.jsonl`, `test_sequences.jsonl`
- Dung `reports/training_summary.json` de xem baseline metric nhanh

## File quan trong nhat
- `catalog/item_metadata_final.jsonl`
- `catalog/item_id_map.json`
- `interactions/mock/user_interactions.jsonl`
- `training/sasrec/train_sequences.jsonl`
- `training/sasrec/valid_metrics.json`
- `training/sasrec/test_metrics.json`
- `demo/recommendations/demo_recommendations.json`
- `demo/ui/demo_config.json`

## Ghi chu quan trong
- `catalog/` moi la source-of-truth cho clips final.
- Khong nen scan toan bo `assets/clips/` de suy ra item list.
- Trong package nay co {clip_file_count} file audio va {len(item_rows)} item final trong catalog.
- Neu so file audio lon hon item final thi phai uu tien dung item trong `catalog/` va `demo/`.
- Neu mot file chua co trong package, xem `manifest.json` de biet dang thieu o dau.
- Co the publish nguyen thu muc nay thanh Kaggle dataset moi.
"""
write_text(readme_text, OUTPUT_ROOT / "README.md")

manifest = {
    "package_name": "ready-data",
    "version": "v1",
    "roots": {
        "ready_items_dataset_root": str(READY_ITEMS_DATASET_ROOT) if READY_ITEMS_DATASET_ROOT else None,
        "ready_items_publish_root": str(READY_ITEMS_ROOT) if READY_ITEMS_ROOT else None,
        "mock_outputs_root": str(MOCK_OUTPUTS_ROOT) if MOCK_OUTPUTS_ROOT else None,
        "sasrec_outputs_root": str(SASREC_OUTPUTS_ROOT) if SASREC_OUTPUTS_ROOT else None,
        "podtok_data_root": str(PODTOK_DATA_ROOT) if PODTOK_DATA_ROOT else None,
        "clip_source_root": str(CLIP_SOURCE_ROOT) if CLIP_SOURCE_ROOT else None,
    },
    "summary": {
        "catalog_items": len(item_rows),
        "catalog_items_with_item_id": sum(1 for row in item_rows if row.get("item_id") is not None),
        "demo_users": len(demo_users),
        "interaction_events": len(interaction_rows),
        "copied_files": len(copied_files),
        "missing_files": len(missing_files),
        "clip_files": clip_file_count,
        "extra_clip_files_vs_catalog": extra_clip_files,
    },
    "copied_files": copied_files,
    "missing_files": missing_files,
    "source_map": source_map,
}
write_json(manifest, OUTPUT_ROOT / "manifest.json")

print(json.dumps(manifest["summary"], indent=2, ensure_ascii=False))
pd.DataFrame(missing_files).head(20)


In [ ]:
required_publish_files = [
    "catalog/item_metadata_final.jsonl",
    "catalog/item_metadata_final.csv",
    "interactions/mock/user_interactions.jsonl",
    "training/sasrec/train_sequences.jsonl",
    "training/sasrec/valid_metrics.json",
    "training/sasrec/test_metrics.json",
    "demo/recommendations/demo_recommendations.json",
    "demo/ui/demo_config.json",
]

recommended_files = [
    "assets/clips",
    "interactions/mock/user_profiles.jsonl",
    "training/sasrec/model_state.pt",
    "reports/training_summary.json",
]

manifest_path = OUTPUT_ROOT / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest chua duoc tao: {manifest_path}")

manifest = orjson.loads(manifest_path.read_bytes())
missing_required = []
missing_recommended = []

for rel_path in required_publish_files:
    if not (OUTPUT_ROOT / rel_path).exists():
        missing_required.append(rel_path)

for rel_path in recommended_files:
    if not (OUTPUT_ROOT / rel_path).exists():
        missing_recommended.append(rel_path)

clip_count = 0
clips_root = OUTPUT_ROOT / "assets" / "clips"
if clips_root.exists():
    clip_count = sum(1 for path in clips_root.rglob("*") if path.is_file())

publish_ready = len(missing_required) == 0
status_text = "READY TO PUBLISH" if publish_ready else "MISSING FILES"

print("=" * 72)
print(status_text)
print("=" * 72)
print("required_missing:", len(missing_required))
print("recommended_missing:", len(missing_recommended))
print("clip_files:", clip_count)
print("catalog_items:", manifest.get("summary", {}).get("catalog_items", 0))
print("interaction_events:", manifest.get("summary", {}).get("interaction_events", 0))
print("demo_users:", manifest.get("summary", {}).get("demo_users", 0))
print()

if missing_required:
    print("Missing required files:")
    for path in missing_required:
        print(" -", path)
    print()

if missing_recommended:
    print("Missing recommended files:")
    for path in missing_recommended:
        print(" -", path)
    print()

print("Publish checklist:")
print(" - Check manifest.json")
print(" - Check demo/recommendations/demo_recommendations.json")
print(" - Check demo/ui/demo_config.json")
print(" - Check assets/clips if web demo needs audio playback")
print(" - Publish whole /kaggle/working/ready-data as a new Kaggle dataset")

pd.DataFrame({
    "group": ["required"] * len(required_publish_files) + ["recommended"] * len(recommended_files),
    "path": required_publish_files + recommended_files,
    "exists": [ (OUTPUT_ROOT / path).exists() for path in required_publish_files + recommended_files ]
})
